# ЭКГ: исправленный препроцессинг и QC

Сначала откройте [локальное сравнение](epi_ecg_preprocessing_comparison.ipynb): оно работает с текущей библиотекой и показывает реальные результаты до/после без обучения моделей.

**Полный эксперимент ниже пока не запускался после исправлений.** Он использует дополнительные API `epi 1.3.0` (ECG_ALIASES, ecg_features_extended, event_metrics и др.), которых нет в локальной `1.2.0`. Эти функции нельзя восстановить по сохранённым выводам ноутбука. Исправлены клетки обработки и QC; остальные части модели сохранены.

Исходный ноутбук со старыми выводами сохранён в `data/reports/preprocessing_v2_20261010/original_notebook.ipynb`. Старые метрики моделей здесь очищены: они не относятся к новой обработке.

Параметры разметки сохранены: окна 300 с, шаг 5 с, конец положительного окна за 300–900 с до приступа. Появилось сохранение точного списка окон. Подавляется полоса 48–52 Гц; исходные отсчёты проверяются на плато и клиппинг до ресэмплинга. QC применяется до нормализации, его маска согласована с индексами меток, пациентов и событий. Пороги, обучаемые по данным, оцениваются только на фоне тренировочной части фолда.

**Метрика в новой таблице:** мощность 48–52 Гц / мощность 0.5–40 Гц, Welch. Она явно отличается от неизвестной функции `line_noise_ratio` из старой версии. `clean_before` и `clean_after` рассчитаны на одних окнах новым определением; `before` означает фильтр локальной версии 1.2.0. Числа нельзя напрямую сопоставлять со старой сохранённой таблицей.


Короткие одиночные выбросы теперь заменяются до ресэмплинга и частотных фильтров.
Сохраняются исходные `x_dev/x_test`, подготовленные `x_dev_prepared/x_test_prepared`
и интервалы замен в `dev_spike_intervals.csv` / `test_spike_intervals.csv`.
Исходный QC по-прежнему запрещает пропуски и насыщение; интерполяция не отменяет
эти причины. Обучаемые пороги QC применяются к подготовленным отсчётам, чтобы
повторно не отклонять уже обработанный короткий выброс по исходному пику.
Пример и ограничения: [замена коротких выбросов](epi_ecg_spike_repair.ipynb).


In [ ]:
from pathlib import Path
import copy
import gc
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from joblib import dump
from sklearn.base import clone
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier

import epi
from epi import Catalog, WindowSet, ECG_ALIASES, ECG_EXTENDED_ALIASES
from epi.evaluation import (
    event_metrics,
    patient_auc_summary,
    patient_folds,
)
from epi.preprocessing import (
    fit_quality_limits,
    line_noise_power_ratio,
    preprocess as preprocess_before,
    ecg_quality_report,
    ecg_raw_quality_report,
    normalize,
    WARNING,
    preprocess_ecg,
    quality_report,
)
from epi.features import (
    ECG_DERIVED_CHANNEL_NAMES,
    ecg_derived_channels,
    ecg_features_extended,
)

try:
    import torch
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
except ImportError:
    torch = None

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "epi/__init__.py").exists())
DATA_DIR = ROOT / "data/epi_data"
CATALOG_DB = ROOT / "data/catalog.sqlite"
OUT_DIR = ROOT / "data/reports/results_ecg_E_300s_lead300_900"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_SPLITS = 5

# Recommended first run.
WINDOW_SEC = 300.0
STRIDE_SEC = 5.0

# Positive lead-time range:
# GAP_BEFORE_SEC <= lead_sec <= GAP_BEFORE_SEC + PREICTAL_SEC
# -> here: 300 s ... 900 s before seizure onset (measured from window end).
GAP_BEFORE_SEC = 300.0
PREICTAL_SEC = 600.0
GAP_AFTER_SEC = 300.0
MERGE_GAP_SEC = 120.0
MIN_OVERLAP_SEC = 5.0

# Prevent one seizure from dominating because of strongly overlapping windows.
MAX_POSITIVE_PER_EVENT = 8

HARD_RATIO = 0.5
HARD_SPAN_SEC = 600.0

TARGET_RATE = 256.0
LOW = 0.5
HIGH = 40.0
NOTCH = 50.0
NORMALIZE_METHOD = "robust"
NORMALIZE_CLIP = None

QC_QUANTILE = 0.995
QC_FIXED = dict(
    max_flat=0.01,
    max_clipping=0.01,
    max_bad_channels=0.25,
    max_nan=0.0,
)

USE_EXTENDED_ECG_ALIASES = False
ECG_SPEC = ECG_EXTENDED_ALIASES if USE_EXTENDED_ECG_ALIASES else ECG_ALIASES

BATCH = 64
RUN_CLASSIC = True
RUN_NEURAL = True
EVALUATE_TEST = True

random.seed(SEED)
np.random.seed(SEED)

print("epi:", epi.__version__)
print("ECG aliases:", ECG_SPEC)
print(
    "forecast lead range:",
    GAP_BEFORE_SEC,
    "...",
    GAP_BEFORE_SEC + PREICTAL_SEC,
    "sec before seizure",
)


In [ ]:
files = sorted(DATA_DIR.rglob("*.e"))
assert DATA_DIR.exists(), DATA_DIR
assert files, f"no .e files in {DATA_DIR}"

print(".e files:", len(files))

catalog = Catalog(CATALOG_DB)
catalog.sync(DATA_DIR, verbose=True)

windows = WindowSet(
    catalog,
    length_sec=WINDOW_SEC,
    stride_sec=STRIDE_SEC,
    min_overlap_sec=MIN_OVERLAP_SEC,
    gap_before_sec=GAP_BEFORE_SEC,
    preictal_sec=PREICTAL_SEC,
    gap_after_sec=GAP_AFTER_SEC,
    merge_gap_sec=MERGE_GAP_SEC,
    rate=TARGET_RATE,
    channels=[ECG_SPEC],
)

coverage = pd.DataFrame(windows.channel_coverage())
coverage["resolved"] = coverage["resolved_channels"].apply(
    lambda xs: ",".join(xs) if xs else None
)

display(coverage[[
    "recording_id", "patient_key", "rel_path", "available", "resolved", "error"
]])

print("\ncoverage:", int(coverage["available"].sum()), "/", len(coverage))
print("selected channel names:")
display(coverage.loc[coverage["available"], "resolved"].value_counts())

coverage.to_csv(OUT_DIR / "ecg_channel_coverage.csv", index=False)

if coverage["available"].mean() < 0.8:
    print(
        "WARNING: generic EKG/ECG coverage is below 80%. "
        "Inspect the table before enabling ECGL/ECGR fallback."
    )


In [ ]:
build_report = windows.build(verbose=True)
split_report = windows.split(seed=SEED, holdout=0.15, verbose=True)
display(pd.DataFrame(split_report))


In [ ]:
def _event_key(row):
    event = row["event_idx"]
    if event is None:
        return (row["recording_id"], row["segment"], "window", row["id"])
    return (row["recording_id"], row["segment"], event)


def balanced_fold_sample(fold, seed):
    positive_rows = list(windows.windows(fold=fold, label=1))
    assert positive_rows, f"{fold}: no positive windows"

    n_events = len({_event_key(row) for row in positive_rows})
    requested_pos = min(
        len(positive_rows),
        max(1, n_events * MAX_POSITIVE_PER_EVENT),
    )

    x, y, groups, metadata = windows.sample(
        fold=fold,
        n=2 * requested_pos,
        positive_ratio=0.5,
        hard_ratio=HARD_RATIO,
        hard_span_sec=HARD_SPAN_SEC,
        seed=seed,
        max_flat=1.0,
        verbose=True,
        return_groups=True,
        return_metadata=True,
        max_positive_per_event=MAX_POSITIVE_PER_EVENT,
    )

    rng = np.random.default_rng(seed)
    pos = np.flatnonzero(y == 1)
    neg = np.flatnonzero(y == 0)
    n = min(len(pos), len(neg))
    if n == 0:
        raise ValueError(f"{fold}: ECG sampling left one class empty")

    keep = np.concatenate([
        rng.choice(pos, n, replace=False),
        rng.choice(neg, n, replace=False),
    ])
    keep = keep[rng.permutation(len(keep))]

    kept_meta = [metadata[i] for i in keep]
    kept_y = y[keep]
    positive_leads = np.array([
        row["lead_sec"]
        for row, label in zip(kept_meta, kept_y)
        if label == 1 and row["lead_sec"] is not None
    ], dtype=float)

    print(
        f"{fold}: positive events={n_events}, "
        f"sampled positive windows={int(kept_y.sum())}"
    )
    if len(positive_leads):
        print(
            f"{fold}: positive lead sec "
            f"min={positive_leads.min():.1f}, "
            f"median={np.median(positive_leads):.1f}, "
            f"max={positive_leads.max():.1f}"
        )

    return (
        x[keep],
        kept_y,
        groups[keep],
        kept_meta,
    )

x_train, y_train, g_train, m_train = balanced_fold_sample("train", SEED)
x_val, y_val, g_val, m_val = balanced_fold_sample("val", SEED + 1)
x_test, y_test, g_test, m_test = balanced_fold_sample("test", SEED + 2)

print("train:", x_train.shape, np.bincount(y_train))
print("val:  ", x_val.shape, np.bincount(y_val))
print("test: ", x_test.shape, np.bincount(y_test))

assert x_train.shape[1] == x_val.shape[1] == x_test.shape[1] == 1


In [ ]:
x_dev = np.concatenate([x_train, x_val], axis=0)
y_dev = np.concatenate([y_train, y_val], axis=0)
g_dev = np.concatenate([g_train, g_val], axis=0)
m_dev = m_train + m_val

print("dev:", x_dev.shape, np.bincount(y_dev), "patients:", len(np.unique(g_dev)))
print("test:", x_test.shape, np.bincount(y_test), "patients:", len(np.unique(g_test)))

cv_splits = list(patient_folds(g_dev, N_SPLITS, y=y_dev, seed=SEED))
for fold, (tr, va) in enumerate(cv_splits, 1):
    print(
        f"fold {fold}: train patients={len(np.unique(g_dev[tr]))}, "
        f"val patients={len(np.unique(g_dev[va]))}, "
        f"train windows={len(tr)}, val windows={len(va)}"
    )


# Save exact window provenance; a seed alone cannot reproduce a changed catalogue.
pd.DataFrame(m_dev).assign(label=y_dev, patient_key=g_dev).to_csv(
    OUT_DIR / "dev_window_manifest.csv", index=False)
pd.DataFrame(m_test).assign(label=y_test, patient_key=g_test).to_csv(
    OUT_DIR / "test_window_manifest.csv", index=False)


In [ ]:
def preprocess_batches(x, metadata):
    from fractions import Fraction
    from scipy.signal import resample_poly
    from epi.preprocessing import repair_ecg_spikes

    assert len(x) == len(metadata)
    prepared = np.empty_like(x)
    source_reports, repair_reports, intervals = [], [], []
    for i, row in enumerate(metadata):
        reader = windows._reader(int(row["recording_id"]))
        reader.select_segment(int(row["segment"]))
        channel = next(ch for alias in ECG_SPEC for ch in reader.channels
                       if ch.upper().replace(" ", "") == alias.upper().replace(" ", ""))
        native = reader.read([channel], row["start_sec"], row["length_sec"]).T
        adc = reader.channel_limits(channel)
        # Preserve source defects before any interpolation or resampling.
        source_reports.append(ecg_raw_quality_report(
            native, reader.sampling_rate, adc_limits=adc, **QC_FIXED))
        repaired, info = repair_ecg_spikes(native, reader.sampling_rate, adc_limits=adc)
        ratio = Fraction(TARGET_RATE / reader.sampling_rate).limit_denominator(10000)
        sampled = resample_poly(repaired, ratio.numerator, ratio.denominator, axis=-1)
        if sampled.shape != x[i].shape:
            raise ValueError("Prepared window shape does not match its metadata")
        prepared[i] = sampled
        repair_reports.append({key: info[key] for key in
                               ["repaired_fraction", "repair_applied", "unrepaired_spike"]})
        for interval in info["intervals"]:
            intervals.append({**interval, "window_index": i,
                "recording_id": int(row["recording_id"]), "segment": int(row["segment"]),
                "window_start_sec": float(row["start_sec"]), "native_rate": reader.sampling_rate,
                "segment_start_sec": row["start_sec"] + interval["start_sec"],
                "segment_stop_sec": row["start_sec"] + interval["stop_sec"]})
        if (i + 1) % 100 == 0:
            print("Native preparation:", i + 1, "/", len(metadata))

    # Native samples have already been repaired; keep that exact mask.
    filtered = np.concatenate([
        preprocess_ecg(prepared[start:start + BATCH], TARGET_RATE,
                       low=LOW, high=HIGH, notch_freq=NOTCH,
                       normalize_signal=False, repair_spikes=False)
        for start in range(0, len(prepared), BATCH)
    ])
    reports = [ecg_quality_report(
        prepared[start:start + BATCH], TARGET_RATE,
        filtered=filtered[start:start + BATCH], repair_spikes=False, **QC_FIXED)
        for start in range(0, len(prepared), BATCH)]
    report = {key: np.concatenate([part[key] for part in reports]) for key in reports[0]}
    for key in ["repaired_fraction", "repair_applied"]:
        report[key] = np.asarray([part[key] for part in repair_reports])
    for i, (source, repair) in enumerate(zip(source_reports, repair_reports)):
        report["keep"][i] &= source["keep"] and not repair["unrepaired_spike"]
        report["fatal"][i] |= source["fatal"]
        report["warning"][i] |= source["warning"] or repair["unrepaired_spike"]
        reasons = set((report["reasons"][i] + ";" + source["reasons"]).split(";")) - {"ok"}
        if repair["unrepaired_spike"]:
            reasons.add("unrepairable_spike")
        report["reasons"][i] = ";".join(sorted(reasons)) or "ok"
    return prepared, filtered, report, intervals


def normalize_batches(x):
    return np.concatenate([
        normalize(x[start:start + BATCH], method=NORMALIZE_METHOD, clip=NORMALIZE_CLIP)
        for start in range(0, len(x), BATCH)
    ])


x_dev_prepared, x_dev_filtered, dev_qc, dev_repairs = preprocess_batches(x_dev, m_dev)
x_test_prepared, x_test_filtered, test_qc, test_repairs = preprocess_batches(x_test, m_test)
x_dev_robust = normalize_batches(x_dev_filtered)
x_test_robust = normalize_batches(x_test_filtered)

# Intervals encode the exact native masks without storing patient signal in Git.
repair_columns = ["window_index", "recording_id", "segment", "window_start_sec", "native_rate",
                  "channel", "start_sample", "stop_sample", "start_sec", "stop_sec",
                  "segment_start_sec", "segment_stop_sec", "repaired", "reason"]
pd.DataFrame(dev_repairs, columns=repair_columns).to_csv(OUT_DIR / "dev_spike_intervals.csv", index=False)
pd.DataFrame(test_repairs, columns=repair_columns).to_csv(OUT_DIR / "test_spike_intervals.csv", index=False)
print("dev basic QC:", int(dev_qc["keep"].sum()), "/", len(x_dev),
      "; windows with interpolation:", int(dev_qc["repair_applied"].sum()))
print("test basic QC:", int(test_qc["keep"].sum()), "/", len(x_test),
      "; windows with interpolation:", int(test_qc["repair_applied"].sum()))


In [ ]:
def noise_batches(x):
    return np.concatenate([
        line_noise_power_ratio(x[start:start + BATCH], TARGET_RATE)
        for start in range(0, len(x), BATCH)
    ])


# This is the explicit NEW power metric, not the unavailable epi 1.3.0 formula.
# Both filters get exactly the same newly loaded windows, before any rejection.
noise_before = np.concatenate([
    line_noise_power_ratio(preprocess_before(
        x_dev[start:start + BATCH], TARGET_RATE, low=LOW, high=HIGH,
        notch_freq=NOTCH, normalize_signal=False), TARGET_RATE)
    for start in range(0, len(x_dev), BATCH)
])
noise_df = pd.DataFrame({
    "patient_key": g_dev, "label": y_dev,
    "raw_power_ratio": noise_batches(x_dev),
    "clean_before": noise_before,
    "clean_after": noise_batches(x_dev_filtered),
    "keep_base_qc": dev_qc["keep"], "qc_reason": dev_qc["reasons"],
})
noise_patient = noise_df.groupby("patient_key").agg(
    n=("label", "size"), raw_median=("raw_power_ratio", "median"),
    raw_p95=("raw_power_ratio", lambda x: np.quantile(x, .95)),
    clean_before=("clean_before", "median"), clean_after=("clean_after", "median"),
    n_keep_base_qc=("keep_base_qc", "sum"),
).sort_values("raw_median", ascending=False)
noise_patient["reduction_factor"] = noise_patient.clean_before / noise_patient.clean_after
print("Metric: Welch P(48–52 Hz) / P(0.5–40 Hz); baseline filter: epi 1.2 preprocess")
print("QC count is before fold-specific trained limits; it is not a quality label for the patient.")
display(noise_patient.head(20))
display(noise_df.groupby("label").agg(n=("label", "size"), kept=("keep_base_qc", "sum")))
noise_patient.to_csv(OUT_DIR / "power_ratio_comparison_by_patient.csv")
noise_df.to_csv(OUT_DIR / "power_ratio_and_qc_by_window.csv", index=False)


In [ ]:
def feature_matrix(x):
    parts = []
    names = None
    for start in range(0, len(x), BATCH):
        values, current_names = ecg_features_extended(
            x[start:start + BATCH],
            TARGET_RATE,
        )
        parts.append(values)
        names = current_names
    return np.vstack(parts), names

X_dev_filtered, feature_names = feature_matrix(x_dev_filtered)
X_test_filtered, _ = feature_matrix(x_test_filtered)
X_dev_robust, feature_names_robust = feature_matrix(x_dev_robust)
X_test_robust, _ = feature_matrix(x_test_robust)

assert feature_names == feature_names_robust

feature_sets_dev = {
    "filtered_ext": X_dev_filtered,
    "robust_ext": X_dev_robust,
}
feature_sets_test = {
    "filtered_ext": X_test_filtered,
    "robust_ext": X_test_robust,
}

print("extended features:", len(feature_names))
print(feature_names[-30:])


In [ ]:
def _base_qc(x):
    if x is x_dev:
        return dev_qc
    if x is x_test:
        return test_qc
    raise ValueError("Pass the original x_dev or x_test array; indices preserve alignment")


def _prepared_qc(x):
    if x is x_dev:
        return x_dev_prepared
    if x is x_test:
        return x_test_prepared
    raise ValueError("Pass the original x_dev or x_test array")


def fit_qc_limits(x, y, indices):
    indices = np.asarray(indices)
    background = indices[(y[indices] == 0) & _base_qc(x)["keep"][indices]]
    if len(background) == 0:
        raise ValueError("QC train split has no acceptable background windows")
    # No validation/test windows enter threshold fitting.
    return fit_quality_limits(_prepared_qc(x)[background], quantile=QC_QUANTILE, **QC_FIXED)


def keep_with_limits(x, indices, limits):
    indices = np.asarray(indices)
    kept = []
    for start in range(0, len(indices), BATCH):
        part = indices[start:start + BATCH]
        report = quality_report(_prepared_qc(x)[part], reject_warnings=WARNING, **limits)
        keep = _base_qc(x)["keep"][part] & np.asarray(report["keep"], dtype=bool)
        kept.extend(part[keep])
    return np.asarray(kept, dtype=int)


def safe_auc(y, score):
    return roc_auc_score(y, score) if len(np.unique(y)) == 2 else np.nan


def metric_row(y, score, threshold=0.5):
    pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {
        "roc_auc": safe_auc(y, score),
        "pr_auc": average_precision_score(y, score) if len(np.unique(y)) == 2 else np.nan,
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "sensitivity": tp / max(tp + fn, 1),
        "specificity": tn / max(tn + fp, 1),
        "threshold": float(threshold),
    }


def choose_threshold(y, score):
    thresholds = np.linspace(0.05, 0.95, 181)
    values = [balanced_accuracy_score(y, score >= t) for t in thresholds]
    return float(thresholds[int(np.argmax(values))])


In [ ]:
def make_classic_models():
    return {
        "logreg C=0.1": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", LogisticRegression(
                C=0.1,
                max_iter=3000,
                class_weight="balanced",
                random_state=SEED,
            )),
        ]),
        "logreg C=1": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", LogisticRegression(
                C=1.0,
                max_iter=3000,
                class_weight="balanced",
                random_state=SEED,
            )),
        ]),
        "extra depth=8": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=3,
                class_weight="balanced",
                n_jobs=-1,
                random_state=SEED,
            )),
        ]),
        "hgb leaves=15": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingClassifier(
                learning_rate=0.03,
                max_leaf_nodes=15,
                l2_regularization=1.0,
                max_iter=300,
                random_state=SEED,
            )),
        ]),
    }


In [ ]:
classic_fold_rows = []
classic_summary_rows = []
classic_oof = {}

if RUN_CLASSIC:
    for variant, X_dev_v in feature_sets_dev.items():
        for model_name, base_model in make_classic_models().items():
            print("\n", variant, model_name)
            oof = np.full(len(y_dev), np.nan, dtype=float)
            fold_aucs = []
            fold_prs = []

            for fold, (tr, va) in enumerate(cv_splits, 1):
                limits = fit_qc_limits(x_dev, y_dev, tr)
                tr_keep = keep_with_limits(x_dev, tr, limits)
                va_keep = keep_with_limits(x_dev, va, limits)

                model = clone(base_model)
                model.fit(X_dev_v[tr_keep], y_dev[tr_keep])
                score = model.predict_proba(X_dev_v[va_keep])[:, 1]
                oof[va_keep] = score

                metrics = metric_row(y_dev[va_keep], score, 0.5)
                fold_events = event_metrics(
                    y_dev[va_keep],
                    score,
                    [m_dev[i] for i in va_keep],
                    threshold=0.5,
                )
                fold_aucs.append(metrics["roc_auc"])
                fold_prs.append(metrics["pr_auc"])
                classic_fold_rows.append({
                    "variant": variant,
                    "model": model_name,
                    "fold": fold,
                    "n_train": len(tr_keep),
                    "n_val": len(va_keep),
                    "train_patients": len(np.unique(g_dev[tr_keep])),
                    "val_patients": len(np.unique(g_dev[va_keep])),
                    "event_sensitivity_0p5": fold_events["event_sensitivity"],
                    **metrics,
                })
                print(
                    f"fold={fold} auc={metrics['roc_auc']:.3f} "
                    f"event_sens={fold_events['event_sensitivity']:.3f} "
                    f"n_val={len(va_keep)}"
                )

            valid = np.isfinite(oof)
            threshold = choose_threshold(y_dev[valid], oof[valid])
            oof_metrics = metric_row(y_dev[valid], oof[valid], threshold)
            oof_patient = patient_auc_summary(
                y_dev[valid], oof[valid], g_dev[valid]
            )
            oof_event = event_metrics(
                y_dev[valid],
                oof[valid],
                [m_dev[i] for i in np.flatnonzero(valid)],
                threshold=threshold,
            )
            classic_oof[(variant, model_name)] = oof

            classic_summary_rows.append({
                "family": "classic",
                "variant": variant,
                "model": model_name,
                "cv_roc_auc_mean": float(np.nanmean(fold_aucs)),
                "cv_roc_auc_std": float(np.nanstd(fold_aucs)),
                "cv_pr_auc_mean": float(np.nanmean(fold_prs)),
                "oof_roc_auc": oof_metrics["roc_auc"],
                "oof_pr_auc": oof_metrics["pr_auc"],
                "oof_balanced_accuracy": oof_metrics["balanced_accuracy"],
                "oof_patient_auc_mean": oof_patient["mean"],
                "oof_event_sensitivity": oof_event["event_sensitivity"],
                "threshold": threshold,
                "n_oof": int(valid.sum()),
            })

classic_folds = pd.DataFrame(classic_fold_rows)
classic_results = pd.DataFrame(classic_summary_rows)
if len(classic_results):
    classic_results = classic_results.sort_values(
        ["cv_roc_auc_mean", "oof_patient_auc_mean"],
        ascending=False,
    ).reset_index(drop=True)

display(classic_results)
classic_folds.to_csv(OUT_DIR / "classic_cv_folds.csv", index=False)
classic_results.to_csv(OUT_DIR / "classic_cv_results.csv", index=False)


In [ ]:
best_classic = None
if RUN_CLASSIC and len(classic_results):
    best_classic = classic_results.iloc[0]
    print(
        "best classic by patient-CV:",
        best_classic["variant"],
        "|",
        best_classic["model"],
        "| AUC",
        round(float(best_classic["cv_roc_auc_mean"]), 4),
    )


In [ ]:
if RUN_NEURAL:
    if torch is None:
        raise ImportError("install torch before RUN_NEURAL=True")

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    AMP = DEVICE.type == "cuda"

    NN_BATCH_SIZE = 16 if AMP else 4
    MAX_EPOCHS = 30
    MIN_EPOCHS = 10
    PATIENCE = 5
    WEIGHT_DECAY = 1e-3
    TRAIN_AUGMENT = True

    MORPH_SEC = 30.0
    MORPH_SAMPLES = int(round(MORPH_SEC * TARGET_RATE))

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    print("device:", DEVICE)
    print("torch:", torch.__version__)
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))
    print("morph branch:", MORPH_SEC, "sec")


In [ ]:
if RUN_NEURAL:
    def derived_batches(x):
        parts = []
    
        for start in range(0, len(x), max(1, BATCH // 2)):
            part = ecg_derived_channels(
                x[start:start + max(1, BATCH // 2)],
                TARGET_RATE,
            )
    
            part = np.nan_to_num(
                part,
                nan=0.0,
                posinf=20.0,
                neginf=-20.0,
            )
    
            part[:, :4] = np.clip(
                part[:, :4],
                -20.0,
                20.0,
            )
    
            parts.append(part.astype(np.float16))
    
        return np.concatenate(parts, axis=0)

    x_dev_derived = derived_batches(x_dev_robust)
    x_test_derived = derived_batches(x_test_robust)

    nn_inputs_dev = {
        "raw": x_dev_robust,
        "derived6": x_dev_derived,
    }
    nn_inputs_test = {
        "raw": x_test_robust,
        "derived6": x_test_derived,
    }

    # Robust-normalized extended features are preferred for wearable transfer.
    AUX_FEATURE_NAMES = list(feature_names_robust)
    X_dev_aux = X_dev_robust.astype(np.float32, copy=False)
    X_test_aux = X_test_robust.astype(np.float32, copy=False)

    print("raw neural input:", x_dev_robust.shape, x_dev_robust.dtype)
    print("derived neural input:", x_dev_derived.shape, x_dev_derived.dtype)
    print("derived channels:", ECG_DERIVED_CHANNEL_NAMES)
    print("aux features:", len(AUX_FEATURE_NAMES))


In [ ]:
if RUN_NEURAL:
    def _groups_for(channels):
        groups = min(8, channels)
        while channels % groups:
            groups -= 1
        return max(groups, 1)


    class ResidualTCNBlock(nn.Module):
        def __init__(self, in_ch, out_ch, stride=1, dilation=1, dropout=0.15):
            super().__init__()
            kernel = 7
            padding = dilation * (kernel // 2)
            self.conv1 = nn.Conv1d(
                in_ch, out_ch, kernel,
                stride=stride,
                padding=padding,
                dilation=dilation,
                bias=False,
            )
            self.norm1 = nn.GroupNorm(_groups_for(out_ch), out_ch)
            self.conv2 = nn.Conv1d(
                out_ch, out_ch, kernel,
                padding=padding,
                dilation=dilation,
                bias=False,
            )
            self.norm2 = nn.GroupNorm(_groups_for(out_ch), out_ch)
            self.dropout = nn.Dropout(dropout)
            self.skip = (
                nn.Identity()
                if in_ch == out_ch and stride == 1
                else nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False)
            )

        def forward(self, x):
            residual = self.skip(x)
            x = F.gelu(self.norm1(self.conv1(x)))
            x = self.dropout(x)
            x = self.norm2(self.conv2(x))
            return F.gelu(x + residual)


    class TCNEncoder(nn.Module):
        def __init__(self, channels, widths=(32, 64, 96), stem_stride=4, dropout=0.15):
            super().__init__()
            first = widths[0]
            self.stem = nn.Sequential(
                nn.Conv1d(channels, first, 15, stride=stem_stride, padding=7, bias=False),
                nn.GroupNorm(_groups_for(first), first),
                nn.GELU(),
            )
            blocks = []
            in_ch = first
            for index, out_ch in enumerate(widths):
                blocks.append(
                    ResidualTCNBlock(
                        in_ch,
                        out_ch,
                        stride=1 if index == 0 else 2,
                        dilation=min(2 ** index, 4),
                        dropout=dropout,
                    )
                )
                in_ch = out_ch
            self.blocks = nn.Sequential(*blocks)
            self.out_dim = 2 * widths[-1]

        def forward(self, x):
            x = self.blocks(self.stem(x))
            avg = F.adaptive_avg_pool1d(x, 1).squeeze(-1)
            mx = F.adaptive_max_pool1d(x, 1).squeeze(-1)
            return torch.cat([avg, mx], dim=1)


    class SmallCNNClassifier(nn.Module):
        def __init__(self, channels=1):
            super().__init__()
            self.encoder = TCNEncoder(
                channels,
                widths=(24, 48, 64),
                stem_stride=8,
                dropout=0.10,
            )
            self.head = nn.Linear(self.encoder.out_dim, 1)

        def forward(self, x, aux=None):
            return self.head(self.encoder(x)).squeeze(1)


    class DerivedTCNClassifier(nn.Module):
        def __init__(self, channels):
            super().__init__()
            self.encoder = TCNEncoder(
                channels,
                widths=(32, 64, 96),
                stem_stride=4,
                dropout=0.15,
            )
            self.head = nn.Sequential(
                nn.Dropout(0.20),
                nn.Linear(self.encoder.out_dim, 1),
            )

        def forward(self, x, aux=None):
            return self.head(self.encoder(x)).squeeze(1)


    class MultiScaleTCNFusionClassifier(nn.Module):
        def __init__(self, channels, aux_dim, morph_samples):
            super().__init__()
            self.morph_samples = morph_samples
            self.long_encoder = TCNEncoder(
                channels,
                widths=(24, 48, 64),
                stem_stride=8,
                dropout=0.15,
            )
            self.morph_encoder = TCNEncoder(
                channels,
                widths=(32, 64, 96),
                stem_stride=4,
                dropout=0.15,
            )
            self.aux = nn.Sequential(
                nn.Linear(aux_dim, 96),
                nn.LayerNorm(96),
                nn.GELU(),
                nn.Dropout(0.20),
                nn.Linear(96, 48),
                nn.GELU(),
            )
            fused_dim = (
                self.long_encoder.out_dim
                + self.morph_encoder.out_dim
                + 48
            )
            self.head = nn.Sequential(
                nn.Linear(fused_dim, 128),
                nn.GELU(),
                nn.Dropout(0.25),
                nn.Linear(128, 1),
            )

        def forward(self, x, aux=None):
            if aux is None:
                raise ValueError("MultiScaleTCNFusionClassifier requires auxiliary features")
            long_repr = self.long_encoder(x)
            short = x[..., -self.morph_samples:]
            morph_repr = self.morph_encoder(short)
            aux_repr = self.aux(aux)
            return self.head(
                torch.cat([long_repr, morph_repr, aux_repr], dim=1)
            ).squeeze(1)


    AUX_DIM = len(AUX_FEATURE_NAMES)
    DERIVED_CH = len(ECG_DERIVED_CHANNEL_NAMES)

    nn_configs = [
        {
            "variant": "raw",
            "model": "small CNN",
            "input_key": "raw",
            "use_aux": False,
            "factory": lambda: SmallCNNClassifier(1),
            "lr": 5e-4,
        },
        {
            "variant": "derived6",
            "model": "residual TCN",
            "input_key": "derived6",
            "use_aux": False,
            "factory": lambda: DerivedTCNClassifier(DERIVED_CH),
            "lr": 4e-4,
        },
        {
            "variant": "derived6+extended_hrv",
            "model": "multiscale TCN fusion",
            "input_key": "derived6",
            "use_aux": True,
            "factory": lambda: MultiScaleTCNFusionClassifier(
                DERIVED_CH,
                AUX_DIM,
                MORPH_SAMPLES,
            ),
            "lr": 3e-4,
        },
    ]


    def count_parameters(model):
        return sum(p.numel() for p in model.parameters() if p.requires_grad)


    for cfg in nn_configs:
        model = cfg["factory"]()
        print(
            cfg["variant"],
            "|",
            cfg["model"],
            "| params:",
            f"{count_parameters(model):,}",
        )
        del model


    def make_aux_preprocessor():
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ])


    def find_nn_config(variant, model):
        for cfg in nn_configs:
            if cfg["variant"] == variant and cfg["model"] == model:
                return cfg
        raise KeyError((variant, model))


    def make_loader(x, y, aux=None, shuffle=False):
        tensors = [
            torch.from_numpy(np.asarray(x)),
            torch.from_numpy(y.astype(np.float32, copy=False)),
        ]
        if aux is not None:
            tensors.append(torch.from_numpy(aux.astype(np.float32, copy=False)))
        ds = TensorDataset(*tensors)
        return DataLoader(
            ds,
            batch_size=NN_BATCH_SIZE,
            shuffle=shuffle,
            num_workers=0,
            pin_memory=AMP,
        )


    def _unpack_nn_batch(batch):
        if len(batch) == 2:
            xb, yb = batch
            auxb = None
        else:
            xb, yb, auxb = batch
        return xb, yb, auxb


    def augment_waveform(x):
        if not TRAIN_AUGMENT:
            return x
        x = x.clone()
        batch, channels, samples = x.shape

        # Lead polarity can differ between devices/montages.
        flip = torch.where(
            torch.rand(batch, 1, 1, device=x.device) < 0.20,
            -torch.ones(batch, 1, 1, device=x.device),
            torch.ones(batch, 1, 1, device=x.device),
        )
        signed_channels = min(channels, 3)
        x[:, :signed_channels] = x[:, :signed_channels] * flip

        # Small sensor noise on waveform-like channels only.
        analog_channels = min(channels, 4)
        if analog_channels:
            noise_scale = torch.rand(batch, 1, 1, device=x.device) * 0.03
            noise = torch.randn(
                batch,
                analog_channels,
                samples,
                device=x.device,
                dtype=x.dtype,
            )
            x[:, :analog_channels] = (
                x[:, :analog_channels] + noise_scale * noise
            )

        # Short missing-signal mask; no time warping because it would alter HRV.
        for index in range(batch):
            if torch.rand((), device=x.device) < 0.20:
                width = int(
                    torch.randint(
                        max(1, int(0.05 * TARGET_RATE)),
                        max(2, int(0.30 * TARGET_RATE)),
                        (1,),
                        device=x.device,
                    ).item()
                )
                if width < samples:
                    start = int(
                        torch.randint(0, samples - width + 1, (1,), device=x.device).item()
                    )
                    x[index, :, start:start + width] = 0
        return x


    def nn_scores(model, x, y, aux=None):
        loader = make_loader(x, y, aux=aux, shuffle=False)
        model.eval()
        ys, ps = [], []
        with torch.no_grad():
            for batch in loader:
                xb, yb, auxb = _unpack_nn_batch(batch)
                xb = xb.to(DEVICE, dtype=torch.float32, non_blocking=True)
                if auxb is not None:
                    auxb = auxb.to(DEVICE, non_blocking=True)
                logits = model(xb, auxb)
                ys.append(yb.numpy())
                ps.append(torch.sigmoid(logits).cpu().numpy())
        return np.concatenate(ys), np.concatenate(ps)


    def train_nn_early(
        model,
        x_train,
        y_train_local,
        x_val,
        y_val_local,
        lr,
        aux_train=None,
        aux_val=None,
    ):
        model = model.to(DEVICE)
        train_loader = make_loader(
            x_train, y_train_local, aux=aux_train, shuffle=True
        )

        positives = float(y_train_local.sum())
        negatives = float(len(y_train_local) - positives)
        pos_weight = torch.tensor(
            [negatives / max(positives, 1.0)],
            device=DEVICE,
        )
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=WEIGHT_DECAY,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
        )
        scaler = torch.amp.GradScaler("cuda", enabled=AMP)

        best_auc = -np.inf
        best_state = None
        best_epoch = 1
        stale = 0

        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            losses = []
            for batch in train_loader:
                xb, yb, auxb = _unpack_nn_batch(batch)
                xb = xb.to(DEVICE, dtype=torch.float32, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                if auxb is not None:
                    auxb = auxb.to(DEVICE, non_blocking=True)

                xb = augment_waveform(xb)

                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=AMP):
                    logits = model(xb, auxb)
                    loss = criterion(logits, yb)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                losses.append(loss.item())

            vy, vp = nn_scores(
                model,
                x_val,
                y_val_local,
                aux=aux_val,
            )
            auc = safe_auc(vy, vp)
            if np.isfinite(auc):
                scheduler.step(auc)

            print(
                f"  epoch={epoch:02d} "
                f"loss={np.mean(losses):.4f} "
                f"val_auc={auc:.4f} "
                f"lr={optimizer.param_groups[0]['lr']:.2e}"
            )

            improved = np.isfinite(auc) and auc > best_auc + 1e-4
            if improved:
                best_auc = auc
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                if epoch >= MIN_EPOCHS:
                    stale = 0
            elif epoch >= MIN_EPOCHS:
                stale += 1

            if epoch >= MIN_EPOCHS and stale >= PATIENCE:
                break

        if best_state is None:
            best_state = copy.deepcopy(model.state_dict())
        model.load_state_dict(best_state)
        return model, best_epoch


    def train_nn_fixed(
        model,
        x_train,
        y_train_local,
        lr,
        epochs,
        aux_train=None,
    ):
        model = model.to(DEVICE)
        loader = make_loader(
            x_train,
            y_train_local,
            aux=aux_train,
            shuffle=True,
        )
        positives = float(y_train_local.sum())
        negatives = float(len(y_train_local) - positives)
        pos_weight = torch.tensor(
            [negatives / max(positives, 1.0)],
            device=DEVICE,
        )
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=lr,
            weight_decay=WEIGHT_DECAY,
        )
        scaler = torch.amp.GradScaler("cuda", enabled=AMP)

        for epoch in range(1, epochs + 1):
            model.train()
            losses = []
            for batch in loader:
                xb, yb, auxb = _unpack_nn_batch(batch)
                xb = xb.to(DEVICE, dtype=torch.float32, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                if auxb is not None:
                    auxb = auxb.to(DEVICE, non_blocking=True)

                xb = augment_waveform(xb)

                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=AMP):
                    logits = model(xb, auxb)
                    loss = criterion(logits, yb)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                losses.append(loss.item())

            print(
                f"  final epoch={epoch:02d}/{epochs} "
                f"loss={np.mean(losses):.4f}"
            )
        return model


In [ ]:
neural_fold_rows = []
neural_summary_rows = []
neural_oof = {}
neural_best_epochs = {}

if RUN_NEURAL:
    for cfg in nn_configs:
        variant = cfg["variant"]
        model_name = cfg["model"]
        input_key = cfg["input_key"]
        use_aux = cfg["use_aux"]
        factory = cfg["factory"]
        lr = cfg["lr"]
        X_wave = nn_inputs_dev[input_key]

        print("\n" + "=" * 90)
        print(f"{variant} | {model_name}")

        oof = np.full(len(y_dev), np.nan, dtype=float)
        fold_aucs, fold_prs, best_epochs = [], [], []

        for fold, (tr, va) in enumerate(cv_splits, 1):
            limits = fit_qc_limits(x_dev, y_dev, tr)
            tr_keep = keep_with_limits(x_dev, tr, limits)
            va_keep = keep_with_limits(x_dev, va, limits)

            aux_train = None
            aux_val = None
            if use_aux:
                aux_pipe = make_aux_preprocessor()
                aux_train = aux_pipe.fit_transform(
                    X_dev_aux[tr_keep]
                ).astype(np.float32)
                aux_val = aux_pipe.transform(
                    X_dev_aux[va_keep]
                ).astype(np.float32)

            print(f"fold {fold}: train={len(tr_keep)} val={len(va_keep)}")
            model, best_epoch = train_nn_early(
                factory(),
                X_wave[tr_keep],
                y_dev[tr_keep],
                X_wave[va_keep],
                y_dev[va_keep],
                lr,
                aux_train=aux_train,
                aux_val=aux_val,
            )

            vy, vp = nn_scores(
                model,
                X_wave[va_keep],
                y_dev[va_keep],
                aux=aux_val,
            )
            oof[va_keep] = vp
            metrics = metric_row(vy.astype(int), vp, 0.5)
            fold_events = event_metrics(
                y_dev[va_keep],
                vp,
                [m_dev[i] for i in va_keep],
                threshold=0.5,
            )
            fold_aucs.append(metrics["roc_auc"])
            fold_prs.append(metrics["pr_auc"])
            best_epochs.append(best_epoch)

            neural_fold_rows.append({
                "variant": variant,
                "model": model_name,
                "fold": fold,
                "best_epoch": best_epoch,
                "n_train": len(tr_keep),
                "n_val": len(va_keep),
                "event_sensitivity_0p5": fold_events["event_sensitivity"],
                **metrics,
            })

            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

        valid = np.isfinite(oof)
        threshold = choose_threshold(y_dev[valid], oof[valid])
        oof_metrics = metric_row(y_dev[valid], oof[valid], threshold)
        oof_patient = patient_auc_summary(
            y_dev[valid], oof[valid], g_dev[valid]
        )
        oof_event = event_metrics(
            y_dev[valid],
            oof[valid],
            [m_dev[i] for i in np.flatnonzero(valid)],
            threshold=threshold,
        )

        key = (variant, model_name)
        neural_oof[key] = oof
        neural_best_epochs[key] = best_epochs

        neural_summary_rows.append({
            "family": "neural",
            "variant": variant,
            "model": model_name,
            "cv_roc_auc_mean": float(np.nanmean(fold_aucs)),
            "cv_roc_auc_std": float(np.nanstd(fold_aucs)),
            "cv_pr_auc_mean": float(np.nanmean(fold_prs)),
            "oof_roc_auc": oof_metrics["roc_auc"],
            "oof_pr_auc": oof_metrics["pr_auc"],
            "oof_balanced_accuracy": oof_metrics["balanced_accuracy"],
            "oof_patient_auc_mean": oof_patient["mean"],
            "oof_event_sensitivity": oof_event["event_sensitivity"],
            "threshold": threshold,
            "n_oof": int(valid.sum()),
            "final_epochs": int(max(1, round(np.median(best_epochs)))),
        })

neural_folds = pd.DataFrame(neural_fold_rows)
neural_results = pd.DataFrame(neural_summary_rows)
if len(neural_results):
    neural_results = neural_results.sort_values(
        ["cv_roc_auc_mean", "oof_patient_auc_mean"],
        ascending=False,
    ).reset_index(drop=True)

display(neural_results)
neural_folds.to_csv(OUT_DIR / "neural_cv_folds.csv", index=False)
neural_results.to_csv(OUT_DIR / "neural_cv_results.csv", index=False)


In [ ]:
candidate_tables = [
    df for df in [classic_results, neural_results]
    if df is not None and len(df)
]
if not candidate_tables:
    raise RuntimeError("no CV results")

all_cv = pd.concat(candidate_tables, ignore_index=True)
all_cv = all_cv.sort_values(
    ["cv_roc_auc_mean", "oof_patient_auc_mean"],
    ascending=False,
).reset_index(drop=True)
all_cv["display_name"] = (
    all_cv["family"].astype(str)
    + " | " + all_cv["variant"].astype(str)
    + " | " + all_cv["model"].astype(str)
)

winner = all_cv.iloc[0]
print("CV winner:")
display(winner.to_frame().T)

# Fit final QC only on development data; then apply frozen limits to test.
final_qc_limits = fit_qc_limits(
    x_dev,
    y_dev,
    np.arange(len(y_dev)),
)
final_dev_keep = keep_with_limits(
    x_dev,
    np.arange(len(y_dev)),
    final_qc_limits,
)
final_test_keep = keep_with_limits(
    x_test,
    np.arange(len(y_test)),
    final_qc_limits,
)

print("final QC dev:", len(final_dev_keep), "/", len(y_dev))
print("final QC test:", len(final_test_keep), "/", len(y_test))

final_model = None
final_aux_preprocessor = None
test_score = None
test_metrics = None
test_event = None
test_patient = None

if EVALUATE_TEST:
    family = winner["family"]
    variant = winner["variant"]
    model_name = winner["model"]
    threshold = float(winner["threshold"])

    if family == "classic":
        final_model = make_classic_models()[model_name]
        final_model.fit(
            feature_sets_dev[variant][final_dev_keep],
            y_dev[final_dev_keep],
        )
        test_score = final_model.predict_proba(
            feature_sets_test[variant][final_test_keep]
        )[:, 1]
        dump(final_model, OUT_DIR / "best_ecg_forecaster.joblib")

    elif family == "neural":
        cfg = find_nn_config(variant, model_name)
        epochs = int(winner["final_epochs"])

        aux_dev = None
        aux_test = None
        if cfg["use_aux"]:
            final_aux_preprocessor = make_aux_preprocessor()
            aux_dev = final_aux_preprocessor.fit_transform(
                X_dev_aux[final_dev_keep]
            ).astype(np.float32)
            aux_test = final_aux_preprocessor.transform(
                X_test_aux[final_test_keep]
            ).astype(np.float32)
            dump(
                final_aux_preprocessor,
                OUT_DIR / "best_ecg_forecaster_aux.joblib",
            )

        final_model = train_nn_fixed(
            cfg["factory"](),
            nn_inputs_dev[cfg["input_key"]][final_dev_keep],
            y_dev[final_dev_keep],
            cfg["lr"],
            epochs,
            aux_train=aux_dev,
        )
        _, test_score = nn_scores(
            final_model,
            nn_inputs_test[cfg["input_key"]][final_test_keep],
            y_test[final_test_keep],
            aux=aux_test,
        )

        torch.save(
            {
                "state_dict": final_model.cpu().state_dict(),
                "variant": variant,
                "model": model_name,
                "input_key": cfg["input_key"],
                "uses_aux": bool(cfg["use_aux"]),
                "aux_feature_names": AUX_FEATURE_NAMES if cfg["use_aux"] else [],
                "derived_channel_names": list(ECG_DERIVED_CHANNEL_NAMES),
                "rate": TARGET_RATE,
                "window_sec": WINDOW_SEC,
                "morph_sec": MORPH_SEC,
                "gap_before_sec": GAP_BEFORE_SEC,
                "preictal_sec": PREICTAL_SEC,
                "lead_min_sec": GAP_BEFORE_SEC,
                "lead_max_sec": GAP_BEFORE_SEC + PREICTAL_SEC,
                "low": LOW,
                "high": HIGH,
                "notch": NOTCH,
                "normalization": "per-window robust",
                "max_positive_per_event": MAX_POSITIVE_PER_EVENT,
            },
            OUT_DIR / "best_ecg_forecaster.pt",
        )
    else:
        raise ValueError(f"unknown family {family!r}")

    test_y = y_test[final_test_keep]
    test_groups = g_test[final_test_keep]
    test_meta = [m_test[i] for i in final_test_keep]

    test_metrics = metric_row(
        test_y,
        test_score,
        threshold,
    )
    test_event = event_metrics(
        test_y,
        test_score,
        test_meta,
        threshold=threshold,
    )
    test_patient = patient_auc_summary(
        test_y,
        test_score,
        test_groups,
    )

    print("test window metrics:", test_metrics)
    print("test sampled-event metrics:", test_event)
    print(
        "test macro patient AUC:",
        test_patient["mean"],
        "valid patients:",
        test_patient["n_valid"],
    )


In [ ]:
display(all_cv)
all_cv.to_csv(OUT_DIR / "all_cv_results.csv", index=False)

summary_rows = []
if EVALUATE_TEST and test_metrics is not None:
    summary_rows.append({
        "family": winner["family"],
        "variant": winner["variant"],
        "model": winner["model"],
        "cv_roc_auc_mean": winner["cv_roc_auc_mean"],
        "cv_roc_auc_std": winner["cv_roc_auc_std"],
        "oof_roc_auc": winner["oof_roc_auc"],
        "oof_pr_auc": winner["oof_pr_auc"],
        "oof_patient_auc_mean": winner["oof_patient_auc_mean"],
        "oof_event_sensitivity": winner["oof_event_sensitivity"],
        "threshold": winner["threshold"],
        "test_roc_auc": test_metrics["roc_auc"],
        "test_pr_auc": test_metrics["pr_auc"],
        "test_balanced_accuracy": test_metrics["balanced_accuracy"],
        "test_sensitivity": test_metrics["sensitivity"],
        "test_specificity": test_metrics["specificity"],
        "test_sampled_event_sensitivity": test_event["event_sensitivity"],
        "test_patient_auc_mean": test_patient["mean"],
        "test_patient_auc_median": test_patient["median"],
    })

summary = pd.DataFrame(summary_rows)
display(summary)
summary.to_csv(OUT_DIR / "final_summary.csv", index=False)

plot_df = all_cv.sort_values("cv_roc_auc_mean")
plt.figure(figsize=(13, max(6, 0.45 * len(plot_df))))
plt.barh(
    plot_df["display_name"],
    plot_df["cv_roc_auc_mean"],
    xerr=plot_df["cv_roc_auc_std"].fillna(0.0),
)
plt.axvline(0.5, linestyle="--")
plt.xlabel("patient-CV ROC-AUC mean ± fold std")
plt.title(
    f"ECG forecasting: {WINDOW_SEC:.0f}s input, "
    f"lead {GAP_BEFORE_SEC:.0f}-{GAP_BEFORE_SEC + PREICTAL_SEC:.0f}s"
)
plt.tight_layout()
plt.savefig(OUT_DIR / "all_cv_roc_auc.png", dpi=160, bbox_inches="tight")
plt.show()

plot_df = all_cv.sort_values("oof_patient_auc_mean")
plt.figure(figsize=(13, max(6, 0.45 * len(plot_df))))
plt.barh(
    plot_df["display_name"],
    plot_df["oof_patient_auc_mean"],
)
plt.axvline(0.5, linestyle="--")
plt.xlabel("OOF macro patient ROC-AUC")
plt.title("Equal-patient view of generalization")
plt.tight_layout()
plt.savefig(OUT_DIR / "all_cv_patient_macro_auc.png", dpi=160, bbox_inches="tight")
plt.show()

with open(OUT_DIR / "qc_limits_final.json", "w", encoding="utf-8") as f:
    json.dump(final_qc_limits, f, indent=2)

with open(OUT_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "seed": SEED,
        "n_splits": N_SPLITS,
        "ecg_aliases": list(ECG_SPEC),
        "extended_aliases": USE_EXTENDED_ECG_ALIASES,
        "window_sec": WINDOW_SEC,
        "stride_sec": STRIDE_SEC,
        "gap_before_sec": GAP_BEFORE_SEC,
        "preictal_sec": PREICTAL_SEC,
        "lead_min_sec": GAP_BEFORE_SEC,
        "lead_max_sec": GAP_BEFORE_SEC + PREICTAL_SEC,
        "gap_after_sec": GAP_AFTER_SEC,
        "merge_gap_sec": MERGE_GAP_SEC,
        "max_positive_per_event": MAX_POSITIVE_PER_EVENT,
        "hard_ratio": HARD_RATIO,
        "hard_span_sec": HARD_SPAN_SEC,
        "rate": TARGET_RATE,
        "low": LOW,
        "high": HIGH,
        "notch": NOTCH,
        "normalization": "per-window robust",
        "qc_quantile": QC_QUANTILE,
        "derived_channels": list(ECG_DERIVED_CHANNEL_NAMES) if RUN_NEURAL else [],
        "aux_features": AUX_FEATURE_NAMES if RUN_NEURAL else [],
        "neural_experiments": [
            {
                "variant": cfg["variant"],
                "model": cfg["model"],
                "params": count_parameters(cfg["factory"]()),
            }
            for cfg in nn_configs
        ] if RUN_NEURAL else [],
        "winner": {
            "family": winner["family"],
            "variant": winner["variant"],
            "model": winner["model"],
            "cv_roc_auc_mean": float(winner["cv_roc_auc_mean"]),
            "threshold": float(winner["threshold"]),
        },
    }, f, indent=2)


In [ ]:
if EVALUATE_TEST and test_score is not None:
    patient_auc = pd.DataFrame(test_patient["per_patient"])
    if len(patient_auc):
        patient_auc = patient_auc.sort_values(
            "roc_auc",
            ascending=False,
            na_position="last",
        ).reset_index(drop=True)
    display(patient_auc)
    patient_auc.to_csv(
        OUT_DIR / "test_patient_auc.csv",
        index=False,
    )

    event_table = pd.DataFrame([test_event])
    display(event_table)
    event_table.to_csv(
        OUT_DIR / "test_sampled_event_metrics.csv",
        index=False,
    )
